In [1]:
import re

def check_password_strength(password):
    """
    Checks the strength of a password based on common criteria:
    - Length: at least 8 characters
    - At least one uppercase letter
    - At least one lowercase letter
    - At least one digit
    - At least one special character
    """
    score = 0
    feedback = []

    # Check length
    if len(password) >= 8:
        score += 1
    else:
        feedback.append("Password should be at least 8 characters long.")

    # Check uppercase
    if re.search(r'[A-Z]', password):
        score += 1
    else:
        feedback.append("Include at least one uppercase letter.")

    # Check lowercase
    if re.search(r'[a-z]', password):
        score += 1
    else:
        feedback.append("Include at least one lowercase letter.")

    # Check digit
    if re.search(r'\d', password):
        score += 1
    else:
        feedback.append("Include at least one number.")

    # Check special character
    if re.search(r'[!@#$%^&*(),.?":{}|<>]', password):
        score += 1
    else:
        feedback.append("Include at least one special character (e.g., !@#$%).")

    # Determine strength
    if score == 5:
        strength = "Strong"
    elif score >= 3:
        strength = "Medium"
    else:
        strength = "Weak"

    return strength, feedback

# Main script for Google Colab
if __name__ == "__main__":
    password = input("Enter your password: ")
    strength, feedback = check_password_strength(password)

    print(f"\nPassword Strength: {strength}")
    if feedback:
        print("Suggestions to improve:")
        for suggestion in feedback:
            print(f"- {suggestion}")
    else:
        print("Your password meets all criteria!")

Enter your password: V33RITOOO

Password Strength: Medium
Suggestions to improve:
- Include at least one lowercase letter.
- Include at least one special character (e.g., !@#$%).


THE ABOVE CODE IS JUST A BASIC VERSION TO UNDERSTAND THE LOGIC AND PROVIDES JUST A BASIC ROADMAP. THE SECOND CODE ON THE 0THER HAND, IS A MORE COMPLE VERSION WITH A PROPER GUI READY FOR DEPLOYMENT.

In [2]:
# Advanced Password Strength Checker - Interactive User Edition
# ========================================================
# This version enhances the previous code with user-friendly guidance:
# - Displays password constraints upfront in clear, formatted text.
# - Uses an interactive ipywidgets GUI for input and evaluation in Colab/Jupyter.
# - After evaluation, shows results with tailored feedback.
# - Retains all enterprise features: rules, logging, config, tests.

import re
import logging
import math
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from ipywidgets import interact, interactive, widgets, Output, VBox, HBox, HTML
from IPython.display import display, Markdown
import unittest

# Configure logging (quiet for interactive use)
logging.basicConfig(level=logging.WARNING, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class PasswordFeedback:
    """Dataclass for structured feedback."""
    strength: str
    score: float  # 0-100 scale
    suggestions: List[str]
    warnings: List[str]

class PasswordRule:
    """Base class for configurable password rules."""
    def __init__(self, name: str, weight: float = 1.0):
        self.name = name
        self.weight = weight

class LengthRule(PasswordRule):
    """Rule for minimum password length."""
    def __init__(self, min_length: int = 8, **kwargs):
        super().__init__(f"Length >= {min_length}", **kwargs)
        self.min_length = min_length

    def validate(self, password: str) -> Tuple[bool, Optional[str]]:
        if len(password) >= self.min_length:
            return True, None
        return False, f"Length should be at least {self.min_length} characters."

class ComplexityRule(PasswordRule):
    """Rule for character set diversity."""
    def __init__(self, **kwargs):
        super().__init__("Character diversity", **kwargs)
        self.patterns = {
            'uppercase': r'[A-Z]',
            'lowercase': r'[a-z]',
            'digits': r'\d',
            'special': r'[!@#$%^&*(),.?":{}|<>]'
        }

    def validate(self, password: str) -> Tuple[int, List[str]]:  # Returns count and failures
        failures = []
        count = 0
        for name, pattern in self.patterns.items():
            if re.search(pattern, password):
                count += 1
            else:
                failures.append(f"Include at least one {name} (e.g., {pattern}).")
        return count >= 2, failures  # At least 2 types for medium strength

class EntropyRule(PasswordRule):
    """Rule based on Shannon entropy (bits of randomness)."""
    def __init__(self, min_entropy: float = 50.0, **kwargs):  # NIST recommends ~60-80 bits
        super().__init__("Entropy", **kwargs)
        self.min_entropy = min_entropy

    def calculate_entropy(self, password: str) -> float:
        """Calculate Shannon entropy."""
        if not password:
            return 0.0
        char_set_size = len(set(password))
        length = len(password)
        if char_set_size == 0:
            return 0.0
        entropy = length * math.log2(char_set_size)
        logger.debug(f"Entropy calculation: length={length}, charset_size={char_set_size}, entropy={entropy:.2f}")
        return entropy

    def validate(self, password: str) -> Tuple[bool, float, Optional[str]]:
        entropy = self.calculate_entropy(password)
        if entropy >= self.min_entropy:
            return True, entropy, None
        return False, entropy, f"Entropy too low ({entropy:.2f} bits; aim for >{self.min_entropy}). Use a mix of random characters."

class DictionaryRule(PasswordRule):
    """Rule to check against common/breached passwords."""
    def __init__(self, common_words: List[str] = None, **kwargs):
        super().__init__("No common words", **kwargs)
        # Mock common passwords (in production, load from file or API)
        self.common_words = common_words or [
            "password", "123456", "qwerty", "letmein", "admin", "welcome", "google", "abc123"
        ]

    def validate(self, password: str) -> Tuple[bool, List[str]]:
        warnings = []
        lower_pwd = password.lower()
        for word in self.common_words:
            if word in lower_pwd or re.search(rf'\b{re.escape(word)}\b', lower_pwd):
                warnings.append(f"Avoid common words like '{word}'.")
        return not warnings, warnings

class PatternRule(PasswordRule):
    """Rule to detect predictable patterns (e.g., sequential chars)."""
    def __init__(self, **kwargs):
        super().__init__("No predictable patterns", **kwargs)
        self.sequential_patterns = [
            r'(?=(?:123|abc|qwe|asd|zxc))',  # Common sequences
            r'(.)\1{2,}',  # Repeated chars
        ]

    def validate(self, password: str) -> Tuple[bool, List[str]]:
        warnings = []
        for pattern in self.sequential_patterns:
            if re.search(pattern, password):
                warnings.append("Avoid sequential (e.g., '123') or repeated characters (e.g., 'aaa').")
                break
        return not warnings, warnings

class AdvancedPasswordChecker:
    """Enterprise-grade password strength checker."""

    def __init__(self, config: Optional[Dict] = None):
        self.rules = [
            LengthRule(min_length=config.get('min_length', 12) if config else 12),  # Stricter default for enterprise
            ComplexityRule(),
            EntropyRule(min_entropy=config.get('min_entropy', 60.0) if config else 60.0),  # NIST-inspired
            DictionaryRule(common_words=config.get('common_words', []) if config else []),
            PatternRule(),
        ]
        self.max_score = sum(rule.weight for rule in self.rules)
        logger.info("AdvancedPasswordChecker initialized with rules.")

    def get_constraints_text(self) -> str:
        """Generate user-friendly constraints text."""
        constraints = [
            f"**Length:** At least {self.rules[0].min_length} characters.",
            f"**Diversity:** Include at least 2 of: uppercase letters (A-Z), lowercase (a-z), digits (0-9), special chars (!@#$%).",
            f"**Entropy:** Aim for >{self.rules[2].min_entropy} bits of randomness (use varied, non-repeating chars).",
            "**No Common Words:** Avoid dictionary words like 'password' or '123456'.",
            "**No Patterns:** Steer clear of sequences (e.g., 'abc123') or repeats (e.g., 'aaa').",
            "*Tip: Generate random passphrases like 'CorrectHorseBatteryStaple42!' for best security.*"
        ]
        return "\n\n".join(constraints)

    def check(self, password: str) -> PasswordFeedback:
        """Perform comprehensive password check."""
        if not isinstance(password, str):
            raise ValueError("Password must be a string.")
        if len(password) == 0:
            raise ValueError("Password cannot be empty.")

        logger.info(f"Checking password of length {len(password)}")
        score = 0.0
        suggestions = []
        warnings = []

        for rule in self.rules:
            if isinstance(rule, LengthRule):
                valid, msg = rule.validate(password)
                if msg:
                    suggestions.append(msg)
                valid = valid
            elif isinstance(rule, ComplexityRule):
                valid, fails = rule.validate(password)
                suggestions.extend(fails)
            elif isinstance(rule, EntropyRule):
                valid, entropy, msg = rule.validate(password)
                if msg:
                    suggestions.append(msg)
            elif isinstance(rule, DictionaryRule):
                valid, dict_warnings = rule.validate(password)
                warnings.extend(dict_warnings)
            elif isinstance(rule, PatternRule):
                valid, pat_warnings = rule.validate(password)
                warnings.extend(pat_warnings)
            else:
                valid = True  # Fallback

            if valid:
                score += rule.weight
            logger.debug(f"Rule '{rule.name}': valid={valid}, score contribution={rule.weight if valid else 0}")

        # Normalize score to 0-100
        normalized_score = (score / self.max_score) * 100

        # Determine strength (NIST-inspired: focus on entropy over composition)
        if normalized_score >= 80:
            strength = "Strong"
        elif normalized_score >= 50:
            strength = "Medium"
        else:
            strength = "Weak"

        feedback = PasswordFeedback(
            strength=strength,
            score=round(normalized_score, 2),
            suggestions=suggestions,
            warnings=warnings
        )

        logger.info(f"Password strength: {strength} (score: {normalized_score:.2f})")
        return feedback

# Interactive GUI for Colab/Jupyter with Constraints
def create_user_gui():
    """Create an interactive widget-based GUI with constraints."""
    # Display constraints as Markdown
    constraints_md = Markdown(checker.get_constraints_text())
    display(constraints_md)

    output = Output()
    password_input = widgets.Password(
        placeholder='Enter your password here...',
        description='Password:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    check_button = widgets.Button(description='Check Strength', button_style='success', layout=widgets.Layout(width='150px'))

    def on_check_click(b):
        with output:
            output.clear_output()
            if not password_input.value:
                print("❌ Please enter a password first!")
                return
            try:
                feedback = checker.check(password_input.value)
                print(f"Results")
                print(f"STRENGTH: {feedback.strength} 💪")
                print(f"SCORE: {feedback.score}/100")

                if feedback.suggestions:
                    print("\nSuggestions to Improve:")
                    for sug in feedback.suggestions:
                        print(f"• {sug}")

                if feedback.warnings:
                    print("\nWarnings:")
                    for warn in feedback.warnings:
                        print(f"⚠️ {warn}")

                if not feedback.suggestions and not feedback.warnings:
                    print("✅ Excellent! Your password meets all enterprise-grade criteria.")

                print("\n---\nRemember: Never share your password. Use a manager for storage.")
            except ValueError as e:
                print(f"❌ Error: {e}")

    check_button.on_click(on_check_click)

    # Layout: Input + Button
    input_box = HBox([password_input, check_button])
    display(VBox([input_box, output]))

# Initialize checker with example config (enterprise-strict)
config = {
    'min_length': 12,
    'min_entropy': 60.0,
    'common_words': ['password', 'google', '123456']
}
checker = AdvancedPasswordChecker(config)

# Run the GUI
create_user_gui()

# Unit Tests (run separately if needed)
class TestPasswordChecker(unittest.TestCase):
    def setUp(self):
        self.checker = AdvancedPasswordChecker()

    def test_weak_password(self):
        feedback = self.checker.check("weak")
        self.assertEqual(feedback.strength, "Weak")
        self.assertLess(feedback.score, 50)

    def test_strong_password(self):
        feedback = self.checker.check("CorrectHorseBatteryStaple42!")
        self.assertEqual(feedback.strength, "Strong")
        self.assertGreaterEqual(feedback.score, 80)

    def test_empty_password(self):
        with self.assertRaises(ValueError):
            self.checker.check("")

    def test_entropy_calculation(self):
        rule = EntropyRule()
        entropy = rule.calculate_entropy("abcABC123!")
        self.assertGreater(entropy, 30)  # Rough check

# To run tests:
suite = unittest.TestLoader().loadTestsFromTestCase(TestPasswordChecker); unittest.TextTestRunner(verbosity=2).run(suite)

**Length:** At least 12 characters.

**Diversity:** Include at least 2 of: uppercase letters (A-Z), lowercase (a-z), digits (0-9), special chars (!@#$%).

**Entropy:** Aim for >60.0 bits of randomness (use varied, non-repeating chars).

**No Common Words:** Avoid dictionary words like 'password' or '123456'.

**No Patterns:** Steer clear of sequences (e.g., 'abc123') or repeats (e.g., 'aaa').

*Tip: Generate random passphrases like 'CorrectHorseBatteryStaple42!' for best security.*

test_empty_password (__main__.TestPasswordChecker.test_empty_password) ... ok
test_entropy_calculation (__main__.TestPasswordChecker.test_entropy_calculation) ... ok
test_strong_password (__main__.TestPasswordChecker.test_strong_password) ... ok
test_weak_password (__main__.TestPasswordChecker.test_weak_password) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.008s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>